In [124]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import entropy
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

plt.style.use(
    '../../Dataset-And-Solve-Class/class-12-ML-how-machine-learns/class-12-deeplearning.mplstyle'
)

In [125]:
# Define paths
ROOT_DIR = "../.." #double back and go to 'Bongo-Dev-DS-AI-ML-Course-Practice-Repo' folder
DATA_DIR = os.path.join(ROOT_DIR, "my-practice/Dataset")
DATASET_PATH = os.path.join(DATA_DIR, "housing.csv")

In [126]:
housing_dataset = pd.read_csv(DATASET_PATH)
housing_dataset.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


### Data Cleaning

In [127]:
housing_dataset.isnull().sum()

price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64

In [128]:
housing_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int64 
 1   area              545 non-null    int64 
 2   bedrooms          545 non-null    int64 
 3   bathrooms         545 non-null    int64 
 4   stories           545 non-null    int64 
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          545 non-null    object
 8   hotwaterheating   545 non-null    object
 9   airconditioning   545 non-null    object
 10  parking           545 non-null    int64 
 11  prefarea          545 non-null    object
 12  furnishingstatus  545 non-null    object
dtypes: int64(6), object(7)
memory usage: 55.5+ KB


### Data Preprocessing, Cleaning & Feature Engineering

In [129]:
housing_dataset.columns.to_list()

['price',
 'area',
 'bedrooms',
 'bathrooms',
 'stories',
 'mainroad',
 'guestroom',
 'basement',
 'hotwaterheating',
 'airconditioning',
 'parking',
 'prefarea',
 'furnishingstatus']

In [130]:
housing_dataset = housing_dataset[[
 'area',
 'bedrooms',
 'bathrooms',
 'stories',
 'mainroad',
 'guestroom',
 'basement',
 'hotwaterheating',
 'airconditioning',
 'parking',
 'prefarea',
 'furnishingstatus',
 'price'
]]

In [131]:
housing_dataset.columns.to_list()

['area',
 'bedrooms',
 'bathrooms',
 'stories',
 'mainroad',
 'guestroom',
 'basement',
 'hotwaterheating',
 'airconditioning',
 'parking',
 'prefarea',
 'furnishingstatus',
 'price']

In [132]:
numerical_cols = housing_dataset.select_dtypes(include='number').columns.drop('price')
categorical_cols = housing_dataset.select_dtypes(include='object').columns

print('numerical_cols = ', numerical_cols.to_list())
print('categorical_cols = ', categorical_cols.to_list())

numerical_cols =  ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']
categorical_cols =  ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea', 'furnishingstatus']


In [133]:
#normalization/standarization
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
housing_dataset[numerical_cols] = scaler.fit_transform(
    housing_dataset[numerical_cols]
)

housing_dataset[numerical_cols].head()


,area,bedrooms,bathrooms,stories,parking
0,1.046726,1.403419,1.421812,1.378217,1.517692
1,1.757010,1.403419,5.405809,2.532024,2.679409
2,2.218232,0.047278,1.421812,0.224410,1.517692
3,1.083624,1.403419,1.421812,0.224410,2.679409
4,1.046726,1.403419,-0.570187,0.224410,1.517692


In [134]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
# housing_dataset[categorical_cols] = le.fit_transform(
#     housing_dataset[categorical_cols]
# ) #it's not work. apply label encoder using for loop or apply() method

# housing_dataset[categorical_cols].apply(le.fit_transform) #also not working

for col in categorical_cols:
    housing_dataset[col] = le.fit_transform(
        housing_dataset[col]
    )    

housing_dataset[categorical_cols].head()

,mainroad,guestroom,basement,hotwaterheating,airconditioning,prefarea,furnishingstatus
0,1,0,0,0,1,1,0
1,1,0,0,0,1,0,0
2,1,0,1,0,0,1,1
3,1,0,1,0,1,1,0
4,1,1,1,0,1,0,0


In [135]:
housing_dataset['basement'].value_counts()

basement
0    354
1    191
Name: count, dtype: int64

In [136]:
#input, output
X = housing_dataset.drop('price', axis=1).values
y = housing_dataset['price'].values 

print(X.shape)
print(y.shape)

(545, 12)
(545,)


In [137]:
#train-test split
from sklearn.model_selection import train_test_split
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_val.shape, y_train_val.shape, X_test.shape, y_test.shape

((436, 12), (436,), (109, 12), (109,))

In [138]:
#train-validation split
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.30, random_state=42
)

In [139]:
print('Train input, output', len(X_train), len(y_train))
print('Train input, output', len(X_val), len(y_val))
print('Train input, output', len(X_test), len(y_test))

Train input, output 305 305
Train input, output 131 131
Train input, output 109 109


### Model Creation, Training & Evaluation

In [140]:
#define model --- need prediction
def model(x, w, b):
    y_pred = np.dot(x,w) + b
    return y_pred

In [141]:
#define cost function --- need to evaluate the model pred
def cost_function(x, y, w, b):
    y_pred = model(x, w, b) 
    mse = np.mean((y - y_pred) ** 2)
    return mse

In [142]:
# define gradient --- need to reduce cost function [find local minima for minimize cost]
def calculate_gradient(x, y, w, b):
    delta = 1e-9
    cost = cost_function(x, y, w, b)
    cost1 = cost_function(x, y, w + delta, b)
    cost2 = cost_function(x, y, w, b + delta)

    dw = (cost1 - cost) / delta
    db = (cost2 - cost) / delta

    return dw, db

In [143]:
# Initialize parameters
np.random.seed(142)
w_init = np.random.randn(X_train.shape[1]) * 0.01
b_init = 0.0

w_init

array([ 0.0012974 ,  0.00902362,  0.01005804,  0.0047189 , -0.00326213,
       -0.0026265 , -0.00368877,  0.01478741,  0.01299381, -0.00525526,
       -0.00536705,  0.00454478])

In [144]:
#Hyperparameters
learning_rate = 0.001
epochs = 50000

In [145]:
# Training the model
def fit(X_train, y_train, w_init, b_init, learning_rate, epochs, X_val = None, y_val = None):
    w = w_init.copy()
    b = b_init
    train_losses, val_losses = [],[]

    for i in range(epochs):
        dw, db = calculate_gradient(X_train, y_train, w, b)
        w -= learning_rate * dw
        b -= learning_rate * db

        if i % 1000 == 0:
            train_losses.append(cost_function(X_train, y_train, w, b))
            if X_val is not None and y_val is not None:
                val_losses.append(cost_function(X_val, y_val, w, b))
            else: 
                val_losses.append(None)
    return w, b, train_losses, val_losses


In [146]:
w_final, b_final, train_losses, val_losses = fit(X_train, y_train, w_init, b_init, learning_rate, epochs, X_val, y_val)

In [147]:
print("Training completed.")
print(f"Final training loss: {train_losses[-1] // 1e9:.2f}")
print(f"Final validation loss: {val_losses[-1]//1e9:.2f}")

Training completed.
Final training loss: 1424.00
Final validation loss: 1512.00


In [150]:
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
#K Fold Cross Validation 

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_mse_scores = []

print("Performing 5-Fold Cross-Validation...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_val)):
    X_fold_train, X_fold_val = X_train_val[train_idx], X_train_val[val_idx] #train, val split - input
    y_fold_train, y_fold_val = y_train_val[train_idx], y_train_val[val_idx] #train, val split - output

    # Re-initialize model
    w_fold = np.random.randn(X_fold_train.shape[1]) * 0.01
    b_fold = 0.0

    # Train
    w_fold, b_fold, _, _ = fit(
        X_fold_train, y_fold_train, w_fold, b_fold, learning_rate, epochs
    )

    # Evaluate
    y_pred_val = X_fold_val @ w_fold + b_fold
    mse = mean_squared_error(y_fold_val, y_pred_val)
    cv_mse_scores.append(mse)
    print(f"  Fold {fold+1} MSE: {mse:.4f}")

print(f"Cross-Validation MSE: {np.mean(cv_mse_scores):.4f} ± {np.std(cv_mse_scores):.4f}")

Performing 5-Fold Cross-Validation...
  Fold 1 MSE: 1665682466445.3857
  Fold 2 MSE: 1211015526317.1760
  Fold 3 MSE: 1335212232393.9968
  Fold 4 MSE: 1730142945913.6179
  Fold 5 MSE: 1226627672620.1179
Cross-Validation MSE: 1433736168738.0591 ± 220844489328.1570
